# gene_statistics.ipynb

- How many ecDNA containing {gene}, in how many cancer types?
- How many unique oncogenes on ecDNA? 158
- How many ecDNA sequences without annotated oncogenes? 96, if partial oncogenes are included; 107 if not.
- What genes are amplified on these?
  - C19MC (5)
  - NTN1 (3)
  - LNX1, near PDGFRA (3)
  - chr17p11.2 (PI3KR6, COX10, CDRT15...) (3)
  - AKT3 (2)
  - chr2q24.2 (no full genes in maximal region) 2)
  - RB1 fusion (2)
### outputs
`out/ecDNA_sans_oncogenes.bed`, required for `recurrent-amps.ipynb`.

In [ ]:
import pandas as pd
import sys
sys.path.append('../../src')
from data_imports import *
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
biosamples = import_biosamples()
genes = import_genes()
genes = genes[genes.sample_name.isin(biosamples[biosamples.in_unique_patient_set].index)]
genes = genes[genes.feature.str.contains("ecDNA")]
genes.head()

In [ ]:
# Most frequently ecDNA-amplified genes
genes.groupby('gene').count().sort_values('sample_name',ascending=False)

In [ ]:
def print_gene_stats(gene):
    df = (genes[(genes.gene==gene)])
    a = len(df)
    b = len(biosamples[biosamples.index.isin(df.sample_name)].cancer_type.unique())
    print(f"{gene} is amplified in {a} ecDNA sequences across {b} tumor types.")
    return

In [ ]:
# How many MYCN ecDNA?
print_gene_stats("MYCN")
# CDK4?
print_gene_stats("CDK4")

In [ ]:
# How many unique oncogenes?
subset = genes[(genes.feature.str.contains("ecDNA"))&(genes.is_canonical_oncogene)]
print(len(subset.gene.unique()))
subset.gene.unique()

In [ ]:
# How many ecDNA sequences without oncogenes?
def import_oncogenes(file='../../data/oncogenes/oncogene_list.txt'):
    with open(file, "r") as f:
        return set(map(str.strip,f.readlines()))

def ecDNA_sans_oncogenes(include_truncated=True):
    df = genes.copy()
    df['ec_uid'] = df.sample_name + '_' + df.amplicon_number + '_' + df.feature
    
    oncogenes = import_oncogenes()
    mask = df.groupby("ec_uid")["is_canonical_oncogene"].transform("all") == False
    df = df[mask]
    if include_truncated:
        df = df[df.groupby("ec_uid")["is_canonical_oncogene"].transform(lambda x: ~x.any())]
    else:
        df['is_full_oncogene'] = df.is_canonical_oncogene & df.truncated.isna()
        df = df[df.groupby("ec_uid")["is_full_oncogene"].transform(lambda x: ~x.any())]
    print(f"{len(df.ec_uid.unique())} ecDNA sequences contain no known oncogene.")
    return df
def count_genes_on_ecDNA_sans_oncogenes(df=None,include_truncated=True):
    if df is None:
        df = ecDNA_sans_oncogenes(include_truncated)
    return df['gene'].value_counts()

df = ecDNA_sans_oncogenes(include_truncated=True)
count_genes_on_ecDNA_sans_oncogenes(df)

In [ ]:
# Write bed of all ecDNA sequences without oncogenes.
import subprocess

outfile = 'out/ecDNA_sans_oncogenes.bed'
beds = list('bed_symlinks/ecDNA_all/'+df.ec_uid.unique()+'_intervals.bed')

cmd = ["bash", "-c",
       rf'''
       for f in "$@"; do
         awk -v fn="$(basename "$f")" 'BEGIN{{OFS="\t"}} {{$4=fn; print}}' "$f"
       done | sort -k1,1 -k2,2n > {outfile}
       ''',
       "--"] + beds

subprocess.run(cmd, check=True)
#print(cmd)